The whole cell does the following:
1. Identifies the Bronze shipments table.
2. Reads it into a Spark DataFrame.
3. Separates business columns from ingestion metadata.
4. Counts the total shipment rows.
5. Defines possible missing-value placeholders.
6. Creates reusable checks for:
   - Real NULL values.
   - Empty or space-only values.
   - Placeholder values such as "N/A".
7. Creates a helper to count rows matching a condition.
8. Creates a helper to combine future profiling reports.
This cell mainly prepares the profiling process. It does not yet produce the complete profiling report, and it does not modify the shipments table.

In [0]:
from functools import reduce
from pyspark.sql import functions as F
#set existing bronz location
CATALOG = "logistics_quality"
BRONZE_SCHEMA = "bronze_layer"

def bronze_table(table_name):
    return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{table_name}`"

#starts with shipment table
shipments = spark.table(bronze_table("shipments"))

#seperate business fields from ingestion metadata. when we ingested the data we added many other colums like _source_row_number, _source_file,_ingested_at

business_columns = [
    column for column in shipments.columns 
    if not column.startswith("_")
]

metadata_columns = [
    column for column in shipments.columns 
    if column.startswith("_")
]

#capture total rows 
total_rows = shipments.count()

if total_rows == 0 :
    raise ValueError("The shipment table is empty. Rewview ingestion first")

#possible representation of missing values 
PLACEHOLDERS = ['null','none','n/a','na']

#this function chekcs whether is the columns has blank values 
#it returns True if the value is empty or only whitspaces, and False otherwise even with null vlaues it return False
def blank_condition(columne_name):
    #this convert column to text
    value = F.col(columne_name).cast("string")
    return F.coalesce(
        value.isNotNull() & value.rlike(r"^\s*$"),
        F.lit(False),
    )

#function to check whether column contain missing-value placeholder or not 
def placeholder_condition(column_name):
    value = F.lower(F.trim(F.col(column_name).cast("string")))

    return F.coalesce(
        value.isin(*PLACEHOLDERS),
        F.lit(False),
    )

# we need function that cmbine both. 
def missing_condition(column_name):
    return(
        F.col(column_name).isNull()
        | blank_condition(column_name)
        | placeholder_condition(column_name)
    )

#count how many rows satisfy a condition 
def count_where(condition):
    return F.coalesce(
        F.sum(F.when(condition,1).otherwise(0)),
        F.lit(0),
    ).cast("long")

#this is helper function to combine all future results 
def combine_reports(reports):
    return reduce(
        lambda first,second:first.unionByName(second),reports
    )

this Table Inspection
- Check the number of rows and columns loaded.
- Review the column names and their current data types.
- Display sample records to confirm that the source data was loaded correctly.
- Business fields are initially stored as strings to preserve the original raw values for later validation and cleaning.

In [0]:
#we create simple datframe that has only one row information about the table we call it table_overview
table_overview = spark.createDataFrame(
    [  (
        #here we define the names of the columns 
        f"{CATALOG}.{BRONZE_SCHEMA}.shipments",
        total_rows,
        len(business_columns),
        len(metadata_columns)
        )
    ],
    #here we define the schema of the whold table or the columns 
    """
    table_name string,
    total_rows long,
    business_columns long,
    metadata_columns long
    """,
)

display(table_overview)

#show stored datatypes 
shipments.printSchema()

#Inspect a asample without changing the data 
display(
    shipments.orderBy("_source_row_number").limit(10)
)

table_name,total_rows,business_columns,metadata_columns
logistics_quality.bronze_layer.shipments,8145,21,3


root
 |-- shipment_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- origin_hub_id: string (nullable = true)
 |-- destination_hub_id: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_type: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- pickup_at: string (nullable = true)
 |-- promised_delivery_at: string (nullable = true)
 |-- delivered_at: string (nullable = true)
 |-- status: string (nullable = true)
 |-- weight_kg: string (nullable = true)
 |-- piece_count: string (nullable = true)
 |-- shipping_fee_egp: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- cod_amount_egp: string (nullable = true)
 |-- delivery_attempts: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_updated_at: string (nullable = true)
 |-- extract_at: string (nullable = true)
 |-- _source_row_number: long (nullable = true)
 |-- _sourc

shipment_id,customer_id,route_id,origin_hub_id,destination_hub_id,service_level,package_type,created_at,pickup_at,promised_delivery_at,delivered_at,status,weight_kg,piece_count,shipping_fee_egp,payment_method,cod_amount_egp,delivery_attempts,source_system,source_updated_at,extract_at,_source_row_number,_source_file,_ingested_at
SHP0000001,C0125,R043,H004,H011,ECONOMY,PARCEL,2026-01-14 15:51:00,2026-01-14 22:39:00,2026-01-20 22:39:00,null,RETURNED,0.6,1,45.97,COD,320.82,1,BRANCH,2026-01-21 02:41:16,2026-07-03 12:00:00,2,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000002,C0138,R002,H001,H003,STANDARD,PARCEL,2026-01-15 07:49:00,2026-13-05 08:00:00,2026-01-18 14:47:33,2026-01-19 05:27:39,DELIVERED,1.43,3,55.62,PREPAID,0,2,OMS_WEB,2026-01-19 05:27:39,2026-07-03 12:00:00,3,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000003,C0129,R097,H009,H010,STANDARD,PARCEL,2026-04-08 18:16:00,2026-04-09 02:19:50,2026-04-13 02:19:50,2026-04-12 08:12:27,DELIVERED,3.21,1,76.37,COD,2835.23,1,OMS_API,2026-04-12 08:12:27,2026-07-03 12:00:00,4,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000004,C0034,R008,H001,H009,STANDARD,PARCEL,2026-04-16 10:54:00,2026-04-16 20:24:31,2026-04-19 20:24:31,2026-04-19 06:59:34,DELIVERED,0.78,1,54.12,PREPAID,0,1,BRANCH,2026-04-19 06:59:34,2026-07-03 12:00:00,5,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000005,C0157,R075,H007,H010,STANDARD,PARCEL,2026-02-24 14:31:00,2026-02-24 19:50:52,2026-02-28 19:50:52,2026-02-26 21:38:01,DELIVERED,1.5,1,65.83,PREPAID,0,1,BRANCH,2026-02-26 21:38:01,2026-07-03 12:00:00,6,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000006,C0048,R100,H010,H001,STANDARD,PARCEL,2026-05-28 07:02:00,2026-05-28 11:45:22,2026-06-01 11:45:22,2026-05-30 22:04:05,DELIVERED,1.08,1,61.01,PREPAID,0,1,OMS_WEB,2026-05-30 22:04:05,2026-07-03 12:00:00,7,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000007,C0120,R046,H005,H002,ECONOMY,DOCUMENT,2026-03-01 16:23:00,2026-03-01 22:19:21,2026-03-06 22:19:21,2026-03-04 23:08:22,DELIVERED,0.24,1,34.54,COD,1280.32,1,OMS_WEB,2026-03-04 23:08:22,2026-07-03 12:00:00,8,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000008,C0022,R033,H003,H012,STANDARD,DOCUMENT,2026-04-10 14:45:00,2026-04-10 17:20:51,2026-04-14 17:20:51,2026-04-13 01:39:52,DELIVERED,N/A,2,59.81,COD,1389.8,1,OMS_API,2026-04-13 01:39:52,2026-07-03 12:00:00,9,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000009,C0023,R055,H005,H012,ECONOMY,parcel,2026-01-20 08:59:00,2026-01-20 17:44:52,2026-01-26 17:44:52,null,RETURNED,2.87,1,59.54,PREPAID,0,1,BRANCH,2026-01-25 02:47:26,2026-07-03 12:00:00,10,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z
SHP0000010,C0116,R041,H004,H009,STANDARD,DOCUMENT,2026-01-26 10:25:00,2026-01-26 18:52:06,2026-01-29 18:52:06,2026-01-28 16:40:54,DELIVERED,0.4,1,52.4,COD,1489.02,1,OMS_WEB,2026-01-28 16:40:54,2026-07-03 12:00:00,11,/Volumes/logistics_quality/bronze_layer/raw_data/shipments.xlsx,2026-10-22T19:40:54.599Z


# Profile missing values and distinct values

For every column, it calculates:
- Number of actual NULL values.
- Number of blank or space-only values.
- Number of placeholder values such as "N/A" or "none".
- Total missing-like values.
- Percentage of missing-like values.
- Number of different populated values.

In [0]:
#empty list to hold calculations for all columns
expressions = []

for index, column in enumerate(business_columns):
    expressions.extend([
        count_where(F.col(column).isNull()).alias(f"null_{index}"),
        count_where(blank_condition(column)).alias(f"blank_{index}"),
        count_where(placeholder_condition(column)).alias(f"placeholder_{index}"),
        F.countDistinct(
            F.when(
                ~missing_condition(column),
                F.col(column),
            )
        ).alias(f"distinct_{index}"),
    ])

#Instead of scanning your large dataset multiple times for each column and metric, Spark computes all of these expressions across all columns in one single pass
counts = shipments.agg(*expressions).first()


# Convert the aggregated counts into one profile row per column.
# Combine nulls, blanks, and placeholders into a missing-like total.
# Calculate the missing-like percentage of all rows.
# Store the results in profile_rows for the final profiling report.
profile_rows = []

for index, column in enumerate(business_columns):
    null_rows = counts[f"null_{index}"]
    blank_rows = counts[f"blank_{index}"]
    placeholder_rows = counts[f"placeholder_{index}"]

    # These three categories are counted separately.
    missing_like_rows = (
        null_rows + blank_rows + placeholder_rows
    )

    profile_rows.append((
        column,
        null_rows,
        blank_rows,
        placeholder_rows,
        missing_like_rows,
        round(missing_like_rows / total_rows * 100, 2),
        counts[f"distinct_{index}"],
    ))

#Creating and Displaying the Profile Table

column_profile = spark.createDataFrame(
    profile_rows,
    """
    column_name string,
    null_rows long,
    blank_rows long,
    placeholder_rows long,
    missing_like_rows long,
    missing_like_pct double,
    distinct_nonmissing_values long
    """,
)

display(
    column_profile.orderBy(
        F.desc("missing_like_pct"),
        "column_name",
    )
)


column_name,null_rows,blank_rows,placeholder_rows,missing_like_rows,missing_like_pct,distinct_nonmissing_values
delivered_at,1593,0,0,1593,19.56,6437
promised_delivery_at,322,0,0,322,3.95,7684
pickup_at,319,0,0,319,3.92,7661
weight_kg,181,0,72,253,3.11,1233
shipping_fee_egp,81,0,0,81,0.99,4705
destination_hub_id,36,0,0,36,0.44,12
cod_amount_egp,25,0,0,25,0.31,5235
shipment_id,21,0,0,21,0.26,7980
created_at,0,0,0,0,0.0,7689
customer_id,0,0,0,0,0.0,161


#Duplicate Profiling
- Compare business columns to find exact duplicate shipment records.
- Group records by shipment_id to find repeated IDs.
- Check whether repeated IDs contain identical or conflicting information.
- Exclude ingestion metadata from comparisons.
- Report the findings without changing the Bronze data.

In [0]:
#apply this only on business columns
business_data = shipments.select(*business_columns)
#count unique rows and duplicate  
unique_business_rows = business_data.distinct().count()
exact_duplicate_extra_rows = (
    total_rows - unique_business_rows
)
#what we do here . we try to check duplicates . but some rows has no ids it's null or missing so we can't consider them duplicated we need to remove them then count duplicated 

#this is dataframe with rows that identified only 
identified_shipments = shipments.filter(
    ~missing_condition("shipment_id")
)

# Find shipment IDs appearing more than once.
repeated_ids = (
    identified_shipments
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
)


# Remove identical business copies temporarily to count distinct versions.
# This does not remove anything from Bronze.
conflicting_ids = (
    identified_shipments
    .select(*business_columns)
    .distinct()
    .groupBy("shipment_id")
    .agg(F.count("*").alias("distinct_business_versions"))
    .filter(F.col("distinct_business_versions") > 1)
)

duplicate_profile = spark.createDataFrame(
    [
        ("exact_duplicate_rows_beyond_first", exact_duplicate_extra_rows),
        ("repeated_shipment_id_groups", repeated_ids.count()),
        ("conflicting_shipment_id_groups", conflicting_ids.count()),
    ],
    "metric string, value long",
)

display(duplicate_profile)


# Show examples of repeated IDs.
display(
    repeated_ids.orderBy(
        F.desc("count"),
        "shipment_id",
    ).limit(20)
)

# Show the different business records for conflicting IDs.
display(
    identified_shipments
    .join(
        conflicting_ids.select("shipment_id"),
        on="shipment_id",
        how="inner",
    )
    .select(*business_columns, "_source_row_number")
    .orderBy("shipment_id", "_source_row_number")
    .limit(30)
)

metric,value
exact_duplicate_rows_beyond_first,100
repeated_shipment_id_groups,144
conflicting_shipment_id_groups,44


shipment_id,count
SHP0000182,2
SHP0000183,2
SHP0000239,2
SHP0000279,2
SHP0000321,2
SHP0000367,2
SHP0000386,2
SHP0000391,2
SHP0000492,2
SHP0000521,2


shipment_id,customer_id,route_id,origin_hub_id,destination_hub_id,service_level,package_type,created_at,pickup_at,promised_delivery_at,delivered_at,status,weight_kg,piece_count,shipping_fee_egp,payment_method,cod_amount_egp,delivery_attempts,source_system,source_updated_at,extract_at,_source_row_number
SHP0000183,C0107,R058,H006,H003,EXPRESS,PARCEL,2026-04-09 14:35:00,2026-04-09 22:33:37,2026-04-11 10:33:37,2026-04-12 02:48:41,DELIVERED,0.84,3,80.64,COD,1197.26,2,OMS_WEB,2026-04-12 02:48:41,2026-07-03 12:00:00,184
SHP0000183,C0107,R058,H006,H003,EXPRESS,PARCEL,2026-04-09 14:35:00,2026-04-09 22:33:37,2026-04-11 10:33:37,2026-04-12 02:48:41,UNKNOWN,0.84,3,80.64,COD,1197.26,2,OMS_WEB,2026-04-12 02:48:41,2026-07-03 12:00:00,8136
SHP0000279,C0086,R129,H012,H008,STANDARD,PARCEL,2026-05-01 08:29:00,2026-05-01 17:19:16,2026-05-05 17:19:16,2026-05-05 05:14:56,DELIVERED,3.06,1,82.42,COD,2487.93,1,BRANCH,2026-05-05 05:14:56,2026-07-03 12:00:00,280
SHP0000279,C0086,R129,H012,H008,STANDARD,PARCEL,2026-05-01 08:29:00,2026-05-01 17:19:16,2026-05-05 17:19:16,2026-05-05 05:14:56,UNKNOWN,3.06,1,82.42,COD,2487.93,1,BRANCH,2026-05-05 05:14:56,2026-07-03 12:00:00,8139
SHP0000719,C0122,R113,H011,H003,ECONOMY,DOCUMENT,2026-01-21 16:38:00,2026-01-21 20:44:13,2026-01-27 20:44:13,2026-01-26 06:51:46,DELIVERED,0.45,1,42.41,COD,1944.22,1,BRANCH,2026-01-26 06:51:46,2026-07-03 12:00:00,720
SHP0000719,C0122,R113,H011,H003,ECONOMY,DOCUMENT,2026-01-21 16:38:00,2026-01-21 20:44:13,2026-01-27 20:44:13,2026-01-26 06:51:46,UNKNOWN,0.45,1,42.41,COD,1944.22,1,BRANCH,2026-01-26 06:51:46,2026-07-03 12:00:00,8117
SHP0001057,C0016,R049,H005,H006,ECONOMY,PARCEL,2026-01-24 17:13:00,2026-01-25 01:19:14,2026-01-30 01:19:14,2026-01-28 23:54:26,DELIVERED,1.99,1,41.77,COD,3184.99,1,OMS_WEB,2026-01-28 23:54:26,2026-07-03 12:00:00,1058
SHP0001057,C0016,R049,H005,H006,ECONOMY,PARCEL,2026-01-24 17:13:00,2026-01-25 01:19:14,2026-01-30 01:19:14,2026-01-28 23:54:26,UNKNOWN,1.99,1,41.77,COD,3184.99,1,OMS_WEB,2026-01-28 23:54:26,2026-07-03 12:00:00,8126
SHP0001493,C0133,R022,H002,H012,STANDARD,PARCEL,2026-01-31 07:34:00,2026-01-31 14:29:08,2026-02-04 14:29:08,null,DELIVERED,1.77,1,74.52,COD,3133.33,1,OMS_API,2026-02-03 07:41:18,2026-07-03 12:00:00,1494
SHP0001493,C0133,R022,H002,H012,STANDARD,PARCEL,2026-01-31 07:34:00,2026-01-31 14:29:08,2026-02-04 14:29:08,null,UNKNOWN,1.77,1,74.52,COD,3133.33,1,OMS_API,2026-02-03 07:41:18,2026-07-03 12:00:00,8105


#Categorical Profiling Approach
- Inspect categorical columns such as status, service_level, and payment_method.
- List each distinct value and count its frequency.
- Display values clearly to reveal spaces, capitalization differences, and spelling issues.
- Record suspicious values for later investigation.
- Do not compare with reference tables or modify the Bronze data at this stage.
- During the validation stage, compare observed values with reference tables.
- Then classify values as valid, aliases to standardize, missing, or invalid.

In [0]:
categorical_columns = [
    "status",
    "service_level",
    "package_type",
    "payment_method",
    "source_system",
]

category_reports=[]
for column in categorical_columns:
    report=(
        shipments
        .groupBy(column)
        .count()
        .select(
            F.lit(column).alias("Column_name"),
            F.col(column).alias("raw_value"),
            F.col("count").alias("row_count"),
            F.to_json(
                F.struct(F.col(column).alias("value")),
                options={"ignoreNullFields":"false"},

            ).alias("value_as_json"),
            F.round(
                F.col("count")/F.lit(total_rows)*100,
            ).alias("row_pct")

        )
    )
    category_reports.append(report)
category_profile = combine_reports(category_reports)
display(category_profile.orderBy("column_name",F.desc("row_pct"),
                                 "raw_value"))


Column_name,raw_value,row_count,value_as_json,row_pct
package_type,PARCEL,6679,"{""value"":""PARCEL""}",82.0
package_type,DOCUMENT,914,"{""value"":""DOCUMENT""}",11.0
package_type,BULKY,507,"{""value"":""BULKY""}",6.0
package_type,parcel,45,"{""value"":""parcel""}",1.0
payment_method,COD,5374,"{""value"":""COD""}",66.0
payment_method,PREPAID,2741,"{""value"":""PREPAID""}",34.0
payment_method,cash on delivery,30,"{""value"":""cash on delivery""}",0.0
service_level,STANDARD,5378,"{""value"":""STANDARD""}",66.0
service_level,ECONOMY,1369,"{""value"":""ECONOMY""}",17.0
service_level,EXPRESS,1317,"{""value"":""EXPRESS""}",16.0


#Numeric Profiling
- Select columns that should contain numbers.
- Temporarily convert their text values to numeric values.
- Count values that fail conversion.
- Calculate valid counts, minimums, maximums, and averages.
- Keep negative and extreme values visible for now.
- Do not correct or remove outliers at this stage.
- Keep the original Bronze data unchanged.

This cell profiles the numeric columns in the Bronze shipments table. It receives the raw shipments DataFrame and produces two reports:
1. A numeric summary for every numeric column.
2. Examples of suspicious numeric values.

Numeric Profile Measures
The report summarizes different conditions in numeric columns:
- Missing-like rows: Values that are NULL, blank, or placeholders such as "N/A".
- Conversion failures: Non-missing text that cannot be converted to a number, such as "unknown".
- Non-finite rows: Values such as NaN, positive infinity, or negative infinity.
- Valid numeric rows: Values successfully converted to usable finite numbers.
- Negative values: Values below zero; these may be invalid for weight, pieces, fees, or attempts and require investigation.
- Zero values: Values equal to zero; zero may be invalid for weight or fees, but may be valid for delivery_attempts or cod_amount_egp.

In [0]:
numeric_columns = [
    "weight_kg",
    "piece_count",
    "shipping_fee_egp",
    "cod_amount_egp",
    "delivery_attempts",
]

numeric_reports = []
numeric_example_reports = []

#The result is a temporary DataFrame containing the original columns plus _missing and _parsed_number, which are used for numeric profiling.
for column in numeric_columns:
    # Create temporary interpretation columns.
    prepared = (
        shipments
        .withColumn("_missing", missing_condition(column)) # crates new column name and  check wheter the column we are tartgeting has any missing condition
        .withColumn(
            "_parsed_number",
            F.expr(f"try_cast(`{column}` AS DOUBLE)"),
        ) # convert the vlaue of the column we are targeting as number 
    )

    #So finite_condition tells us which values are safe to use in numeric calculations such as averages and minimums.This checks that the parsed value is:Not NULL.Not NaN.Not positive or negative infinity.Stores the result as True or False.
    finite_condition = F.coalesce(
        F.col("_parsed_number").isNotNull()
        & ~F.isnan("_parsed_number")
        & ~F.col("_parsed_number").isin(
            float("inf"),
            float("-inf"),
        ),
        F.lit(False),
    )
    #this numeric value Keeps the number only when it is finite. Otherwise, it stores NULL. 
    prepared = (
        prepared
        .withColumn("_finite", finite_condition)
        .withColumn(
            "_numeric_value",
            F.when(
                F.col("_finite"),
                F.col("_parsed_number"),
            ),
        )
    )

# Summarize parsing results and the raw numeric distribution.
    report = (
        prepared
        .agg(
            count_where(
                F.col("_missing")
            ).alias("missing_like_rows"),

            count_where(
                ~F.col("_missing")
                & F.col("_parsed_number").isNull()
            ).alias("parse_failure_rows"),

            count_where(
                ~F.col("_missing")
                & F.col("_parsed_number").isNotNull()
                & ~F.col("_finite")
            ).alias("nonfinite_rows"),

            F.count("_numeric_value").alias("finite_numeric_rows"),

            count_where(
                F.col("_numeric_value") < 0
            ).alias("negative_rows"),

            count_where(
                F.col("_numeric_value") == 0
            ).alias("zero_rows"),

            F.min("_numeric_value").alias("minimum"),
            F.max("_numeric_value").alias("maximum"),
            F.avg("_numeric_value").alias("mean"),

            F.percentile_approx(
                "_numeric_value",
                [0.25, 0.50, 0.75],
                10000,
            ).alias("quartiles_q1_median_q3"),
        )
        .withColumn("column_name", F.lit(column))
    )

    numeric_reports.append(report)

    # Collect examples of parsing problems and negative values.
    # A negative value is a finding here; formal rules come later.
    examples = (
        prepared
        .filter(
            ~F.col("_missing")
            & (
                ~F.col("_finite")
                | (F.col("_parsed_number") < 0)
            )
        )
        .select(
            F.lit(column).alias("column_name"),
            "shipment_id",
            F.col(column).alias("raw_value"),
            "_source_row_number",
        )
        .orderBy("_source_row_number")
        .limit(10)
    )

    numeric_example_reports.append(examples)

numeric_profile = combine_reports(numeric_reports)
numeric_examples = combine_reports(numeric_example_reports)

display(numeric_profile.orderBy("column_name"))
display(numeric_examples.orderBy("column_name", "_source_row_number"))

missing_like_rows,parse_failure_rows,nonfinite_rows,finite_numeric_rows,negative_rows,zero_rows,minimum,maximum,mean,quartiles_q1_median_q3,column_name
25,0,0,8120,30,2727,-200.0,3499.82,1169.2716046798032,"List(0.0, 900.48, 2156.54)",cod_amount_egp
0,0,0,8145,0,1012,0.0,99.0,1.3931246163290363,"List(1.0, 1.0, 1.0)",delivery_attempts
0,0,0,8145,25,0,-1.0,3.0,1.5969920196439533,"List(1.0, 1.0, 2.0)",piece_count
81,30,0,8034,21,0,-85.0,250000.0,888.7846713965642,"List(56.22, 65.72, 82.74)",shipping_fee_egp
253,50,0,7842,30,20,-4.5,9999.0,49.60902065799541,"List(0.97, 1.72, 3.06)",weight_kg


column_name,shipment_id,raw_value,_source_row_number
cod_amount_egp,SHP0000158,-200,159
cod_amount_egp,SHP0000378,-200,379
cod_amount_egp,SHP0000588,-200,589
cod_amount_egp,SHP0000733,-200,734
cod_amount_egp,SHP0000826,-200,827
cod_amount_egp,SHP0001147,-200,1148
cod_amount_egp,SHP0001310,-200,1311
cod_amount_egp,SHP0001348,-200,1349
cod_amount_egp,SHP0001901,-200,1902
cod_amount_egp,SHP0002048,-200,2049


#Date and Timestamp Profiling
- Select columns that should contain dates or timestamps.
- Try to parse ISO and day/month/year formats.
- Count missing values and parsing failures.
- Count successfully parsed timestamps.
- Identify the earliest and latest dates.
- Keep the original timestamp strings unchanged.
- Check logical date relationships later during validation.


In [0]:
#sets the default timezone for the entire Spark session to UTC.
spark.conf.set("spark.sql.session.timeZone","UTC")

date_columns = [
    "created_at",
    "pickup_at",
    "promised_delivery_at",
    "delivered_at",
    "source_updated_at",
    "extract_at",
]

date_reports = []
date_example_reports = []
for column in date_columns:
    prepared= (
        shipments
        .withColumn("_missing",missing_condition(column))
        .withColumn(
            "_default_timestamp",
            F.expr(f"try_cast(`{column}` AS TIMESTAMP)"),
        )  #This code tries to read a date value using default timestamp format. if it not able to read the value it returns null 
        .withColumn(
            "_dmy_timestamp",
            F.coalesce(
                F.expr(
                    f"try_to_timestamp(`{column}`, "
                    "'dd/MM/yyyy HH:mm:ss')"
                ),
                F.expr(
                    f"try_to_timestamp(`{column}`, "
                    "'dd/MM/yyyy HH:mm')"
                ),
                F.expr(
                    f"try_to_timestamp(`{column}`, "
                    "'dd/MM/yyyy')"
                ),
            ),
        ) #here we try some another documented formats 
        #next we will return the first format that already parsed 
        .withColumn(
            "_parsed_timestamp",
            F.coalesce(
                "_default_timestamp",
                "_dmy_timestamp",
            ),
        )
    )
    #  creates one summary row for the current date column.
    report = (
        prepared
        .agg(
            count_where(
                F.col("_missing")
            ).alias("missing_like_rows"),

            count_where(
                ~F.col("_missing")
                & F.col("_default_timestamp").isNotNull()
            ).alias("default_format_rows"),

            count_where(
                ~F.col("_missing")
                & F.col("_default_timestamp").isNull()
                & F.col("_dmy_timestamp").isNotNull()
            ).alias("dmy_format_rows"),

            count_where(
                ~F.col("_missing")
                & F.col("_parsed_timestamp").isNull()
            ).alias("parse_failure_rows"),

            F.min("_parsed_timestamp").alias("earliest_timestamp"),
            F.max("_parsed_timestamp").alias("latest_timestamp"),
        )
        .withColumn("column_name", F.lit(column))
    )

    date_reports.append(report)

    # Show populated values that could not be parsed.
    examples = (
        prepared
        .filter(
            ~F.col("_missing")
            & F.col("_parsed_timestamp").isNull()
        )
        .select(
            F.lit(column).alias("column_name"),
            "shipment_id",
            F.col(column).alias("raw_value"),
            "_source_row_number",
        )
        .orderBy("_source_row_number")
        .limit(10)
    )

    date_example_reports.append(examples)

date_profile = combine_reports(date_reports)
date_examples = combine_reports(date_example_reports)

display(date_profile.orderBy("column_name"))
display(date_examples.orderBy("column_name", "_source_row_number"))

missing_like_rows,default_format_rows,dmy_format_rows,parse_failure_rows,earliest_timestamp,latest_timestamp,column_name
0,8088,31,26,2026-01-01T07:02:00.000Z,2027-01-01T00:00:00.000Z,created_at
1593,6552,0,0,2026-01-01T12:02:00.000Z,2026-06-30T23:58:00.000Z,delivered_at
0,8145,0,0,2026-07-03T12:00:00.000Z,2026-07-03T12:00:00.000Z,extract_at
319,7798,0,28,2026-01-01T11:28:50.000Z,2026-06-30T20:19:22.000Z,pickup_at
322,7823,0,0,2026-01-02T23:32:15.000Z,2026-07-06T14:37:38.000Z,promised_delivery_at
0,8145,0,0,2025-12-01T00:00:00.000Z,2026-06-30T23:58:24.000Z,source_updated_at


column_name,shipment_id,raw_value,_source_row_number
created_at,SHP0000035,2026-02-30 09:00:00,36
created_at,SHP0000165,2026-02-30 09:00:00,166
created_at,SHP0000647,2026-02-30 09:00:00,648
created_at,SHP0001129,2026-02-30 09:00:00,1130
created_at,SHP0001215,2026-02-30 09:00:00,1216
created_at,SHP0001298,2026-02-30 09:00:00,1299
created_at,SHP0001959,2026-02-30 09:00:00,1960
created_at,SHP0002105,2026-02-30 09:00:00,2106
created_at,SHP0002751,2026-02-30 09:00:00,2752
created_at,SHP0003489,2026-02-30 09:00:00,3490
